# Notebook 4: Graph analytics in NetworkX

Import the Python libraries.

In [1]:
import re
import sys

import networkx as nx
import polars as pl
import watermark

import kuzu
#import ryugraph

Run a "watermark" to show which library versions are used in this notebook's runtime environment.

In [2]:
%load_ext watermark
%watermark
%watermark --iversions

Last updated: 2025-11-18T07:16:08.346091+00:00

Python implementation: CPython
Python version       : 3.13.8
IPython version      : 9.1.0

Compiler    : Clang 17.0.0 (clang-1700.0.13.3)
OS          : Darwin
Release     : 24.6.0
Machine     : arm64
Processor   : arm
CPU cores   : 14
Architecture: 64bit

networkx : 3.4.2
re       : 2.2.1
polars   : 1.29.0
watermark: 2.5.0
kuzu     : 0.9.0
sys      : 3.13.8 (main, Oct  7 2025, 12:01:51) [Clang 17.0.0 (clang-1700.0.13.3)]



## Four-Step Design Pattern

Reconnect to the existing graph database.

In [3]:
DB_PATH: str = "./db"

if "kuzu" in sys.modules:
    db: kuzu.Database = kuzu.Database(DB_PATH)
    conn: kuzu.Connection = kuzu.Connection(db)
else:
    db: ryugraph.Database = ryugraph.Database(DB_PATH)
    conn: ryugraph.Connection = ryugraph.Connection(db)

There is a popular **four-step design pattern** used with investigative graphs:

 - Step 1: run _entity resolution_ to merge the structured data sources and generate graph elements
 - Step 2: partition the graph to _identify subgraphs_ as potential fraud networks -- typically with graph algorithms such as _Louvain partitioning_ or _strongly connected components_
 - Step 3: centrality to _rank individuals_ of interest within each subgraph -- to identify the most connected elements of a subgraph, as a most likely controlling party or ultimate beneficial owner (UBO)
 - Step 4: process these subgraphs through *case management* tools

## Graph algorithms in NetworkX

Here we can use a Cypher query to approximate _connected components_.
We'll take this short-cut, extracting subgraphs of resolved entities and data records from OpenSanctions and Open Ownership, plus the relations among them.

We'll transfer to [`NetworkX`](https://networkx.org/) to analyze the subgraphs as a [`NetworkX.MultiDiGraph`](https://networkx.org/documentation/stable/reference/classes/multidigraph.html)

In [4]:
subgraphs: nx.MultiDiGraph = conn.execute("""
MATCH (a:Entity:OpenOwnership:OpenSanctions)-[b]->(c:Entity:OpenOwnership:OpenSanctions)
RETURN *
""").get_as_networkx(directed = True)

Using [_betweenness centrality_](https://networkx.org/documentation/stable/reference/algorithms/generated/networkx.algorithms.centrality.betweenness_centrality.html) is a way to show important "bridge" nodes within a subgraph -- think of this as "focusing a lens" on a vicinity of the graph.


In [5]:
tween: dict[ str, float ] = nx.betweenness_centrality(subgraphs)
scale: float = max(tween.values())

Store the results into a dataframe, while scaling the centrality measures to `[0.0, 1.0]` so they can be used for enhancing graph visualization.

In [6]:
df_tween: pl.DataFrame = pl.DataFrame(
    {
        "id": re.sub(r"^\w+\_", "", id),
        "betweenness_centrality": rank / scale,
    }
    for id, rank in tween.items()
).sort("betweenness_centrality", descending = True)

df_tween.head(5)

id,betweenness_centrality
str,f64
"""sz:60""",1.0
"""sz:249""",0.574122
"""sz:104""",0.52251
"""sz:47""",0.49831
"""sz:96""",0.447653


Add a new property `betweenness_centrality` as a new column to the relevant tables.

**Since this next step modifies the graph database schema, it can only be run ONCE** -- otherwise you'll need to go back and re-run the previous notebook `3.reform.ipynb` first.

In [7]:
conn.execute("ALTER TABLE Entity ADD betweenness_centrality FLOAT");
conn.execute("ALTER TABLE OpenOwnership ADD betweenness_centrality FLOAT");
conn.execute("ALTER TABLE OpenSanctions ADD betweenness_centrality FLOAT");

In [8]:
conn.execute(
    f"""
    LOAD FROM df_tween
    MERGE (s1:Entity {{id: id}})
    SET s1.betweenness_centrality = betweenness_centrality
    MERGE (s2:OpenSanctions {{id: id}})
    SET s2.betweenness_centrality = betweenness_centrality
    MERGE (s3:OpenOwnership {{id: id}})
    SET s3.betweenness_centrality = betweenness_centrality
    """
);

Let's query to the see the top-ranked entities in a particular subgraph.

In [9]:
res = conn.execute("""
MATCH (c:Entity)-[b]->(a:Entity)
WHERE c.descrip CONTAINS "Abassin"
RETURN a.id, a.descrip, a.betweenness_centrality
ORDER BY a.betweenness_centrality DESC
LIMIT 10
""")

res.get_as_pl()

a.id,a.descrip,a.betweenness_centrality
str,str,f32
"""sz:155""","""BARLLOWS SERVICES LTD""",0.011279
"""sz:2""","""LMAR GB LTD""",0.011279
"""sz:9""","""BARLLOWS SERVICES LTD""",0.011279
"""sz:99""","""WELLHANCIA HEALTH CARE LTD""",0.011279
"""sz:156""","""Rehana Badshah""",0.010715
"""sz:ds_open-ownership_172078534…","""sz:ds_open-ownership:172078534…",0.0
"""sz:ds_open-ownership_674754810…","""sz:ds_open-ownership:674754810…",0.0
"""sz:ds_open-sanctions_NK-25vyVF…","""sz:ds_open-sanctions:NK-25vyVF…",0.0


Finally, close the database connection.

---

In [10]:
db.close()

---